# Node2vec baseline

Use the user–subreddit graph to predict the five Big Five scores.
Install `node2vec` in the notebook environment and run the cells in order.

In [ ]:
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from node2vec import Node2Vec
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

DATA = Path("../data/preprocessed")  # From the project's src folder
OUTPUT = Path("../data/node2vec_baseline")
DIMENSIONS = 64
SEED = 42

TRAITS = ["agreeableness", "openness", "conscientiousness",
          "extraversion", "neuroticism"]
ALPHAS = [0.01, 0.1, 1, 10, 100]

users = pd.read_csv(DATA / "main/users.csv").set_index("author")
outer = pd.read_csv(DATA / "outer_folds.csv").set_index("author")
inner = pd.read_csv(DATA / "inner_folds.csv")
assert users.index.is_unique and outer.index.is_unique
assert set(users.index) == set(outer.index)

OUTPUT.mkdir(parents=True, exist_ok=True)

## 1. Build the graph and get user vectors

Add an edge when a training user has commented in a subreddit. All edges
have equal weight. Node2vec learns vectors for the nodes in this graph.

A held-out user has no node in the training graph. For both training and
held-out users, we average the learned vectors of their known subreddits.
This gives both groups the same type of feature. Ignore unknown subreddits.
Use a zero vector if none are known, and record how often this happens.

In [ ]:
counts = pd.read_csv(DATA / "main/subreddit_counts.csv")
subreddits = counts.groupby("author")["subreddit"].agg(list)
assert set(subreddits.index) == set(users.index)


def make_vectors(train_users, test_users):
    graph = nx.Graph()
    for author in train_users:
        for subreddit in subreddits[author]:
            graph.add_edge(f"user:{author}", f"sub:{subreddit}")

    walker = Node2Vec(
        graph, dimensions=DIMENSIONS, walk_length=20, num_walks=10,
        p=1, q=0.5, workers=1, seed=SEED,
    )
    model = walker.fit(window=5, min_count=1, batch_words=4096, seed=SEED)

    vectors = np.zeros((len(train_users) + len(test_users), DIMENSIONS))
    known_counts = []
    for i, author in enumerate(list(train_users) + list(test_users)):
        known = [f"sub:{name}" for name in subreddits[author] if f"sub:{name}" in model.wv]
        known_counts.append(len(known))
        if known:
            vectors[i] = model.wv[known].mean(axis=0)

    split = len(train_users)
    return vectors[:split], vectors[split:], known_counts[split:]

## 2. Tune and test Ridge

Train a separate graph for each inner split. Use the same vectors to try
all alpha values and all traits in that split. Pick alpha by average
validation RMSE for each trait, then train a new graph on all outer
training users and predict the outer test users.

In [ ]:
data = users[TRAITS].join(outer)
predictions = []
coverage = []

for fold in sorted(data["outer_fold"].unique()):
    train = data[data["outer_fold"] != fold]
    test = data[data["outer_fold"] == fold]
    inner_fold = inner[inner["outer_fold"] == fold].set_index("author")["inner_fold"]
    assert inner_fold.index.is_unique and set(inner_fold.index) == set(train.index)
    inner_fold = inner_fold.reindex(train.index)

    errors = pd.DataFrame(0.0, index=ALPHAS, columns=TRAITS)
    for val_fold in sorted(inner_fold.unique()):
        inner_train = train[inner_fold != val_fold]
        validation = train[inner_fold == val_fold]
        X_train, X_val, _ = make_vectors(inner_train.index, validation.index)

        for alpha in ALPHAS:
            ridge = Ridge(alpha=alpha, solver="lsqr")
            ridge.fit(X_train, inner_train[TRAITS])
            predicted = ridge.predict(X_val)
            rmse = mean_squared_error(validation[TRAITS], predicted, multioutput="raw_values") ** 0.5
            errors.loc[alpha] += rmse

    best_alpha = errors.idxmin()
    X_train, X_test, known_counts = make_vectors(train.index, test.index)
    coverage.append(pd.DataFrame({
        "author": test.index,
        "outer_fold": fold,
        "known_subreddits": known_counts,
    }))

    for trait in TRAITS:
        ridge = Ridge(alpha=best_alpha[trait], solver="lsqr")
        ridge.fit(X_train, train[trait])
        predictions.append(pd.DataFrame({
            "author": test.index,
            "outer_fold": fold,
            "trait": trait,
            "true": test[trait].to_numpy(),
            "node2vec_prediction": ridge.predict(X_test),
            "mean_prediction": train[trait].mean(),
            "alpha": best_alpha[trait],
        }))

    print("Finished fold", fold)

## 3. Check the results

Each user has one test prediction per trait. Compute the metrics using those
predictions. Pearson correlation is left blank if the predictions are constant.

In [ ]:
results = pd.concat(predictions, ignore_index=True)
assert len(results) == len(users) * len(TRAITS)
assert not results.duplicated(["author", "trait"]).any()

metrics = []
for trait, rows in results.groupby("trait"):
    actual = rows["true"]
    for model in ["node2vec", "mean"]:
        predicted = rows[f"{model}_prediction"]
        correlation = float("nan")
        if predicted.nunique() > 1 and actual.nunique() > 1:
            correlation = pearsonr(actual, predicted)[0]
        metrics.append({
            "trait": trait,
            "model": model,
            "rmse": mean_squared_error(actual, predicted) ** 0.5,
            "pearson_r": correlation,
            "r2": r2_score(actual, predicted),
        })

metrics = pd.DataFrame(metrics)
results.to_csv(OUTPUT / "predictions.csv", index=False)
metrics.to_csv(OUTPUT / "metrics.csv", index=False)
coverage = pd.concat(coverage, ignore_index=True)
coverage.to_csv(OUTPUT / "coverage.csv", index=False)
print("Test users with no known subreddits:", coverage["known_subreddits"].eq(0).sum())
display(metrics.round(3))